# v8 generated-C training from `ck.nn`

This notebook authors a bounded FP32 dense/GQA experiment in Python. The existing v8 workflow performs generated-C forward, backward, AdamW, checkpoint/resume, PyTorch oracle comparison, visualization, and independent inference export. Python does not execute replacement model arithmetic.

In [ ]:
from pathlib import Path
import json, sys
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'version' / 'v8').is_dir(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'version' / 'v7'))
import ckernel_engine as cke
RUN_DIR = ROOT / 'version' / 'v8' / '.cache' / 'python_authoring' / 'english_fixture'

In [ ]:
model = cke.models.qwen3_tiny(vocab=384, dim=32, layers=4, hidden=64, heads=4, kv_heads=2, context_len=32, dtype='float32')
experiment = cke.v8.compile(
    model, run_name='v8-python-english-fixture', run_dir=RUN_DIR,
    dataset=cke.v8.DatasetConfig(corpus=ROOT / 'version/v8/training/english_byte_v1.json', max_train_tokens=320, max_validation_tokens=64),
    tokenizer=cke.v8.TokenizerConfig(kind='bpe', vocab_size=384),
    training=cke.v8.TrainingConfig(epochs=2, grad_accum=2),
)
print(model)
print('parameters:', model.parameter_count())
experiment.graph.to_markdown()

In [ ]:
preflight = experiment.preflight()
print(preflight['status'], experiment.preflight_path)
[(row['label'], [candidate['provider'] for candidate in row['candidates']]) for row in preflight['capabilities']]

In [ ]:
EXECUTE = False  # Set True for generated-C training, parity, resume, visualization, and export.
report = experiment.run() if EXECUTE else None
print('command:', ' '.join(experiment.command()))
print('report:', experiment.report_path)

In [ ]:
if report:
    checks = report.get('checks', {})
    parity = checks.get('pytorch_trajectory', {})
    trajectory = parity.get('trajectory', [])
    print('status:', report['status'])
    print('dataset/tokenizer:', json.dumps(report.get('corpus', {}), indent=2))
    print('gradient max abs diff:', parity.get('max_gradient_abs_diff'))
    print('weight max abs diff:', parity.get('max_weight_abs_diff'))
    print('checkpoint/resume:', checks.get('fresh_process_resume'))
    print('inference export:', checks.get('inference_export'))
    print('IR visualizer:', RUN_DIR / 'ir_report.html')
    try:
        import matplotlib.pyplot as plt
        if trajectory:
            plt.plot([row.get('microstep') for row in trajectory], [row.get('cke_loss') for row in trajectory], label='CKE')
            plt.plot([row.get('microstep') for row in trajectory], [row.get('torch_loss') for row in trajectory], label='PyTorch')
            plt.xlabel('microstep'); plt.ylabel('loss'); plt.legend(); plt.show()
    except ImportError:
        print('matplotlib unavailable; inspect trajectory in', experiment.report_path)